In [18]:
from wp import WPLinear
import utils
from net import PerturbNet, BPNet
import torch
import numpy as np
lr = 1e-10
sigma = 1e-10
num_perts = 1
momentum = 0.001
dampening = 0

device="cpu"

train_loader, test_loader, in_shape, out_shape = utils.construct_dataloaders(
    "cifar10", 128, device, validation=True
)



in_shape = np.prod(in_shape)  # for linear networks
dist_sampler = utils.make_dist_sampler(
            "normal",
            device,
        )


torch.manual_seed(42)


# network with momentum


# network with meta learning

# dataset cifar10

# loss func

Files already downloaded and verified


In [13]:
train_loader

### Model definition + set up

In [19]:
# network definition

model_momentum = torch.nn.Sequential(
    torch.nn.Flatten(),
    WPLinear(
        in_shape,
        500,
        bias=False,
        pert_type="FFD",
        dist_sampler=dist_sampler,
        sigma=sigma,
        num_perts=num_perts,
        device=device,
        zero_masking=False,
        orthogonal_perts=False,
        mu_scaling_factor=0,
        meta_lr=0,
    ),
    torch.nn.ReLU(),
    WPLinear(
        500,
        500,
        bias=False,
        pert_type="FFD",
        dist_sampler=dist_sampler,
        sigma=sigma,
        num_perts=num_perts,
        device=device,
        zero_masking=False,
        orthogonal_perts=False,
        mu_scaling_factor=0,
        meta_lr=0,
    ),
    torch.nn.ReLU(),
    WPLinear(
        500,
        out_shape,
        bias=False,
        pert_type="FFD",
        dist_sampler=dist_sampler,
        sigma=sigma,
        num_perts=num_perts,
        device=device,
        zero_masking=False,
        orthogonal_perts=False,
        mu_scaling_factor=0,
        meta_lr=0,
    ),
)

network_mom = PerturbNet(
    network=model_momentum,
    num_perts=1,
    pert_type="FFD",
    BP_network=None,
)


model_meta = torch.nn.Sequential(
    torch.nn.Flatten(),
    WPLinear(
        in_shape,
        500,
        bias=False,
        pert_type="FFD_meta",
        dist_sampler=dist_sampler,
        sigma=sigma,
        num_perts=num_perts,
        device=device,
        zero_masking=False,
        orthogonal_perts=False,
        mu_scaling_factor=dampening,
        meta_lr=momentum,
    ),
    torch.nn.ReLU(),
    WPLinear(
        500,
        500,
        bias=False,
        pert_type="FFD_meta",
        dist_sampler=dist_sampler,
        sigma=sigma,
        num_perts=num_perts,
        device=device,
        zero_masking=False,
        orthogonal_perts=False,
        mu_scaling_factor=dampening,
        meta_lr=momentum,
    ),
    torch.nn.ReLU(),
    WPLinear(
        500,
        out_shape,
        bias=False,
        pert_type="FFD_meta",
        dist_sampler=dist_sampler,
        sigma=sigma,
        num_perts=num_perts,
        device=device,
        zero_masking=False,
        orthogonal_perts=False,
        mu_scaling_factor=dampening,
        meta_lr=momentum,
    ),
)

network_meta = PerturbNet(
    network=model_momentum,
    num_perts=1,
    pert_type="FFD_meta",
    BP_network=None,
)

In [20]:
# network with momentum


non_updated_weights = []
weights_momentum = []
weights_meta = []

for name, param in model_momentum.named_parameters():
    if name.endswith("mu") or name.endswith("sigma"):
        non_updated_weights.append(param)
    else:
        weights_momentum.append(param)

for name, param in model_meta.named_parameters():
    if name.endswith("mu") or name.endswith("sigma"):
        non_updated_weights.append(param)
    else:
        weights_meta.append(param)

momentum_optimizer = torch.optim.SGD(
    weights_momentum,
    lr,
    momentum=momentum,
    dampening=dampening,
    nesterov=False,
)

meta_model_optimizer = torch.optim.SGD(
    weights_meta,
    lr,
    momentum=0,
    dampening=0,
    nesterov=False,
)

loss_obj = torch.nn.CrossEntropyLoss(reduction="none")
loss_func = lambda input, target, onehot: loss_obj(input, target)

In [37]:
def train(
    model,
    device,
    train_loader,
    optimizer,
    loss_func,
    log_interval=1,
    loud=False,
    num_perts = 1,
    batch_size=128,
    num_classes=10,
):
    """
    Trains model for one epoch

    Parameters
    ----------
    loud : bool
        If True, prints average loss and accuracy

    log_interval : int
        Determines the number of batches between the logging of accuracy
    """

    model.train()
    i = 0 
    for batch_idx, (data, target) in enumerate(train_loader):

        if(i>3):
            return train_results
        optimizer.zero_grad()
        onehots = (
            torch.nn.functional.one_hot(target, num_classes).to(device).to(data.dtype)
        )
        data, target = data.to(device), target.to(device)

        loss = model.train_step( data, target, onehots, loss_func)

        optimizer.step()

        loss, output= model.test_step(data, target, onehots, loss_func)

        pred = output.argmax(dim=1, keepdim=True)

        correct = pred.eq(target.view_as(pred)).sum().item()

        if (batch_idx % log_interval == 0) and loud:
            print(
                "Train batch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}".format(
                    i,
                    correct,
                    batch_size, #batch size
                    100.0 * correct / batch_size,
                    loss,
                )
            )

        loss /= len(train_loader.dataset)
        train_results = [loss, (100.0 * correct / batch_size) ]
        i += 1

    return train_results

In [38]:
print("Training of meta model")

train(network_meta, device, train_loader, meta_model_optimizer, loss_func, 1, True)

print("Training of momentum model")
train(network_mom, device, train_loader, momentum_optimizer, loss_func, 1, True)

Training of meta model
Train batch: 0 [15/128 (12%)]	Loss: 296.674194
Train batch: 1 [7/128 (5%)]	Loss: 296.413330
Train batch: 2 [6/128 (5%)]	Loss: 297.047211
Train batch: 3 [8/128 (6%)]	Loss: 299.390686
Training of momentum model
Train batch: 0 [14/128 (11%)]	Loss: 296.229767
Train batch: 1 [12/128 (9%)]	Loss: 294.143555
Train batch: 2 [6/128 (5%)]	Loss: 299.865540
Train batch: 3 [9/128 (7%)]	Loss: 297.746368


[0.007443659210205078, 7.03125]